# Module 9: RAG & LLM Applications
## Vector Databases: 03 - Meta FAISS (Facebook AI Similarity Search)

### Theoretical Overview
FAISS is an industry-standard C++ library with Python bindings for billion-scale similarity search:
- `IndexFlatL2` / `IndexFlatIP`: Exact brute-force search.
- `IndexIVFFlat`: Inverted File Index dividing space into Voronoi cells. Scans only top $n_{probe}$ cells.
- `IndexIVFPQ`: Inverted file with Product Quantization, reducing RAM consumption by $8\times - 16\times$.

In [ ]:
import numpy as np

class MockFAISSIndexIVF:
    """Simulates Voronoi clustering and nprobe parameter in FAISS IVF."""
    def __init__(self, nlist: int = 4):
        self.nlist = nlist
        self.centroids = None
        self.invlists = {i: [] for i in range(nlist)}
        
    def train(self, data: np.ndarray):
        # Pick random centroids
        indices = np.random.choice(len(data), self.nlist, replace=False)
        self.centroids = data[indices].copy()
        
    def add(self, data: np.ndarray):
        for i, vec in enumerate(data):
            # Assign to closest centroid
            dists = np.sum((self.centroids - vec) ** 2, axis=1)
            cluster_id = int(np.argmin(dists))
            self.invlists[cluster_id].append((i, vec))
            
    def search(self, query: np.ndarray, k: int = 3, nprobe: int = 2):
        # Find closest centroids to query
        c_dists = np.sum((self.centroids - query) ** 2, axis=1)
        probe_clusters = np.argsort(c_dists)[:nprobe]
        
        candidates = []
        for cid in probe_clusters:
            for idx, vec in self.invlists[cid]:
                d = np.sum((vec - query) ** 2)
                candidates.append((idx, d))
        candidates.sort(key=lambda x: x[1])
        return candidates[:k]

np.random.seed(42)
data = np.random.randn(200, 16)
ivf = MockFAISSIndexIVF(nlist=5)
ivf.train(data)
ivf.add(data)
matches = ivf.search(data[0], k=3, nprobe=2)
print("FAISS IVF Top Search Matches (Index, Distance):", matches)